In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.petropulse_ops.operational_alerts (
  alert_id STRING,
  region_id STRING,
  region_name STRING,
  event_type STRING COMMENT 'seismic, weather, or combined',
  severity STRING COMMENT 'Elevated or High',
  risk_score DOUBLE,
  message STRING,
  status STRING COMMENT 'OPEN or RESOLVED',
  created_at TIMESTAMP,
  resolved_at TIMESTAMP,
  resolved_by STRING,
  resolution_note STRING
);

In [0]:
def classify_event_type(seismic_score, wind_score, precip_score):
    weather = max(wind_score, precip_score)
    if seismic_score >= 30 and weather >= 30:
        return "combined"
    if seismic_score >= weather:
        return "seismic"
    return "weather"

def generate_alert_message(region_name, event_type, severity, risk_score,
                           nearby_events, max_magnitude, wind_kmh, precip_mm):
    drivers = {
        "seismic": f"{nearby_events} seismic events within range, max magnitude {max_magnitude:.1f}",
        "weather": f"wind {wind_kmh:.0f} km/h, precipitation {precip_mm:.1f} mm",
        "combined": (f"{nearby_events} seismic events (max M{max_magnitude:.1f}) "
                     f"with wind {wind_kmh:.0f} km/h and precipitation {precip_mm:.1f} mm"),
    }[event_type]
    return f"[{severity.upper()}] {region_name}: {event_type} risk, score {risk_score:.1f}. {drivers}."

In [0]:
import uuid
from pyspark.sql import functions as F

scores = spark.table("workspace.petropulse_gold.region_risk_scores").collect()
open_ids = {r.region_id for r in spark.table("workspace.petropulse_ops.operational_alerts")
            .filter("status = 'OPEN'").collect()}

new_rows = []
for s in scores:
    if s.risk_band in ("Elevated", "High") and s.region_id not in open_ids:
        et = classify_event_type(s.seismic_score, s.wind_score, s.precip_score)
        msg = generate_alert_message(s.region_name, et, s.risk_band, s.risk_score,
                                     s.nearby_events, s.max_magnitude,
                                     s.wind_speed_kmh, s.precipitation_mm)
        new_rows.append((str(uuid.uuid4()), s.region_id, s.region_name, et,
                         s.risk_band, float(s.risk_score), msg, "OPEN"))

if new_rows:
    schema = "alert_id string, region_id string, region_name string, event_type string, severity string, risk_score double, message string, status string"
    (spark.createDataFrame(new_rows, schema)
       .withColumn("created_at", F.current_timestamp())
       .withColumn("resolved_at", F.lit(None).cast("timestamp"))
       .withColumn("resolved_by", F.lit(None).cast("string"))
       .withColumn("resolution_note", F.lit(None).cast("string"))
       .write.mode("append").saveAsTable("workspace.petropulse_ops.operational_alerts"))
print("Alerts opened:", len(new_rows))

In [0]:
%sql
UPDATE workspace.petropulse_ops.operational_alerts
SET status = 'RESOLVED', resolved_at = current_timestamp(),
    resolved_by = 'system', resolution_note = 'Risk band returned to Low'
WHERE status = 'OPEN'
  AND region_id IN (SELECT region_id FROM workspace.petropulse_gold.region_risk_scores WHERE risk_band = 'Low');